# EEdit — Composition Evaluation (2 images/group subset)

Runs **reference-guided composition** on a **2-image subset** of each TF-ICON group (Real-Cartoon, Real-Painting, Real-Sketch, Real-Real) and computes evaluation metrics.

Input data is downloaded automatically from your Google Drive input folder.  
Results are uploaded immediately after generation to your Google Drive output folder.

After the install cell, **restart the runtime once**, then run all cells in order.

In [1]:
!nvidia-smi || true

!git clone https://github.com/yuriYanZeXuan/EEdit.git /content/EEdit || true
%cd /content/EEdit
!git fetch --all
!git checkout 35e95e3

%pip uninstall -y numpy opencv-python opencv-python-headless || true
%pip install -q --upgrade pip
%pip install -q --upgrade --force-reinstall --no-cache-dir \
  numpy==1.26.4 \
  torch==2.5.1 torchvision==0.20.1 xformers==0.0.28.post3 \
  diffusers==0.31.0 transformers==4.46.1 accelerate==1.1.0 \
  sentencepiece==0.2.0 safetensors==0.5.2 huggingface_hub==0.26.2 \
  ftfy==6.3.1 einops==0.8.1 omegaconf==2.3.0 \
  pillow==11.1.0 opencv-python-headless==4.10.0.84 gdown \
  lpips torchmetrics

print("Install complete. Restart the runtime now, then continue.")


Thu May 28 10:09:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os
os.kill(os.getpid(), 9)


In [1]:
import numpy, torch, transformers
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")


numpy: 1.26.4
torch: 2.5.1+cu124
transformers: 4.46.1
CUDA: True NVIDIA A100-SXM4-40GB


In [2]:
# Patch composition_gen.py: use weights/transformer/ dir for config.
%cd /content/EEdit
import re

HELPER = """
def _resolve_transformer_config(weights_dir):
    import os, shutil
    dir_path = os.path.join(weights_dir, "transformer")
    if os.path.isfile(os.path.join(dir_path, "config.json")):
        return dir_path
    fallback_dir = os.path.join(weights_dir, "transformer_config_dir")
    os.makedirs(fallback_dir, exist_ok=True)
    dst = os.path.join(fallback_dir, "config.json")
    if not os.path.exists(dst):
        shutil.copy2(os.path.join(weights_dir, "transformer_config.json"), dst)
    return fallback_dir

"""

script = "composition_gen.py"
path = f"/content/EEdit/{script}"
text = open(path).read()
if "_resolve_transformer_config" in text:
    print(f"{script}: already patched")
else:
    new_text = re.sub(
        r"config\s*=\s*[^\n,]*transformer_config\.json[^\n,]*",
        "config=_resolve_transformer_config(args.weights_dir)",
        text
    )
    new_text = new_text.replace("def load_models(", HELPER + "def load_models(", 1)
    open(path, "w").write(new_text)
    print(f"Patched {script}")


/content/EEdit
Patched composition_gen.py


In [3]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from datetime import datetime
from pathlib import Path
import os

auth.authenticate_user()
drive_service = build("drive", "v3")

INPUT_FILE_ID    = "1U7BIJZZinAzraAt_T8jAKX3uPa5c-HQa"
OUTPUT_FOLDER_ID = "1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q"

_ts = datetime.now().strftime("%Y%m%d_%H%M")
RUN_FOLDER_ID = None

def _get_run_folder():
    global RUN_FOLDER_ID
    if RUN_FOLDER_ID is None:
        RUN_FOLDER_ID = _gdrive_mkdir(drive_service, f"EEdit_comp_subset_{_ts}", OUTPUT_FOLDER_ID)
        print(f"Created run folder: EEdit_comp_subset_{_ts}")
        print(f"View: https://drive.google.com/drive/folders/1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q")
    return RUN_FOLDER_ID

def _gdrive_mkdir(service, name, parent_id):
    meta = {"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]}
    return service.files().create(body=meta, fields="id").execute()["id"]

def _gdrive_upload_file(service, local_path, parent_id):
    meta = {"name": os.path.basename(local_path), "parents": [parent_id]}
    media = MediaFileUpload(local_path, resumable=True)
    service.files().create(body=meta, media_body=media, fields="id").execute()

def _upload_tree(service, local_dir, parent_id):
    total = 0
    for item in sorted(Path(local_dir).iterdir()):
        if item.is_dir():
            child_id = _gdrive_mkdir(service, item.name, parent_id)
            total += _upload_tree(service, str(item), child_id)
        else:
            _gdrive_upload_file(service, str(item), parent_id)
            total += 1
    return total

def upload_task_results(task_name, generated_dir, originals_dir=None):
    task_id = _gdrive_mkdir(drive_service, task_name, _get_run_folder())
    total = 0
    if os.path.isdir(generated_dir):
        gen_id = _gdrive_mkdir(drive_service, "generated", task_id)
        n = _upload_tree(drive_service, generated_dir, gen_id)
        print(f"  generated/: {n} files")
        total += n
    if originals_dir and os.path.isdir(originals_dir):
        orig_id = _gdrive_mkdir(drive_service, "originals", task_id)
        n = _upload_tree(drive_service, originals_dir, orig_id)
        print(f"  originals/: {n} files")
        total += n
    print(f"[{task_name}] uploaded {total} files total.")
    return task_id

print("Google Drive authenticated. Upload helpers ready.")
print(f"Output folder: https://drive.google.com/drive/folders/1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q")


Google Drive authenticated. Upload helpers ready.
Output folder: https://drive.google.com/drive/folders/1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q


In [4]:
import os, shutil, zipfile
from pathlib import Path
from googleapiclient.http import MediaIoBaseDownload

COMPOSITION_DIR = "/content/TF-ICON/inputs"
os.makedirs(COMPOSITION_DIR, exist_ok=True)

def gdrive_download(service, file_id, dest_path):
    req = service.files().get_media(fileId=file_id)
    with open(dest_path, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=64*1024*1024)
        done = False
        while not done:
            status, done = dl.next_chunk()
            print(f"  {status.progress()*100:.0f}%", end="\r")
    print(f"  Downloaded: {os.path.basename(dest_path)}")

def extract_zip(zip_path, dest_dir):
    tmp = zip_path + "_tmp"
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(tmp)
    items = [x for x in os.listdir(tmp) if x != "__MACOSX"]
    src = (os.path.join(tmp, items[0])
           if len(items) == 1 and os.path.isdir(os.path.join(tmp, items[0]))
           else tmp)
    for item in os.listdir(src):
        dst = os.path.join(dest_dir, item)
        if os.path.exists(dst):
            shutil.rmtree(dst) if os.path.isdir(dst) else os.remove(dst)
        shutil.move(os.path.join(src, item), dst)
    shutil.rmtree(tmp, ignore_errors=True)

# Download the single composition ZIP directly by file ID
FILE_ID = "1U7BIJZZinAzraAt_T8jAKX3uPa5c-HQa"
meta = drive_service.files().get(fileId=FILE_ID, fields="name,size").execute()
fname = meta["name"]
mb    = int(meta.get("size", 0)) / 1024**2
print(f"Downloading {fname}  ({mb:.1f} MB) ...")
local_zip = f"/content/{fname}"
gdrive_download(drive_service, FILE_ID, local_zip)
print(f"Extracting to {COMPOSITION_DIR} ...")
extract_zip(local_zip, COMPOSITION_DIR)
os.remove(local_zip)

n = sum(1 for _ in Path(COMPOSITION_DIR).rglob("*") if _.is_file())
print(f"\nComposition input: {n} files in {COMPOSITION_DIR}")
if n == 0:
    raise RuntimeError("No composition files extracted. Check the file ID or ZIP contents.")

# Show folder structure
for p in sorted(Path(COMPOSITION_DIR).iterdir()):
    if p.is_dir():
        count = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"  {p.name}/  ({count} files)")


  Downloaded: composition-test.zip
Extracting to /content/TF-ICON/inputs ...

Composition input: 53 files in /content/TF-ICON/inputs
  Real-Cartoon/  (13 files)
  Real-Painting/  (13 files)
  Real-Real/  (13 files)
  Real-Sketch/  (13 files)


## 1. Hugging Face Weights

Accept the FLUX.1-dev access agreement first: https://huggingface.co/black-forest-labs/FLUX.1-dev

Then run this cell with your HF token.

In [6]:
from huggingface_hub import login, snapshot_download
from getpass import getpass
import os

token = getpass("Hugging Face token: ")
login(token=token)

os.makedirs("/content/EEdit/weights", exist_ok=True)

snapshot_download(
    repo_id="black-forest-labs/FLUX.1-dev",
    local_dir="/content/EEdit/weights",
    allow_patterns=[
        "flux1-dev.safetensors",
        "transformer/config.json",
        "transformer_config.json",
        "model_index.json",
        "scheduler/*",
        "text_encoder/*",
        "text_encoder_2/*",
        "tokenizer/*",
        "tokenizer_2/*",
        "vae/*",
    ],
)
print("Weights downloaded.")


Hugging Face token: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.53G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/820 [00:00<?, ?B/s]

Weights downloaded.


## 2. Composition Setup — 2 images per group

Organises the TF-ICON data from `/content/TF-ICON/inputs/` into per-group image configs.  
**Only the first 2 samples per group are used** (Real-Cartoon, Real-Painting, Real-Sketch, Real-Real).

In [7]:
import os, re, json, shutil, numpy as np
from pathlib import Path
from PIL import Image

SUBSET_PER_GROUP = 2

tf_root    = Path("/content/TF-ICON/inputs")
eedit_root = Path("/content/EEdit/input/composition")
cfg_root   = Path("/content/EEdit/configs/composition")
eedit_root.mkdir(parents=True, exist_ok=True)

groups = {"Real-Cartoon": [], "Real-Painting": [], "Real-Sketch": [], "Real-Real": []}

def pick(files, patterns, exclude=()):
    for pat in patterns:
        for f in files:
            if re.match(pat, f.name.lower()) and not any(e in f.name.lower() for e in exclude):
                return f
    return None

for group_name in ["Real-Cartoon", "Real-Painting", "Real-Sketch", "Real-Real"]:
    domain_path = tf_root / group_name
    if not domain_path.exists(): continue
    collected = 0
    for idx, sample_dir in enumerate(sorted(domain_path.iterdir())):
        if collected >= SUBSET_PER_GROUP: break
        if not sample_dir.is_dir(): continue
        prompt = sample_dir.name
        files  = [p for p in sample_dir.iterdir() if p.is_file()]
        bg          = pick(files, [r"^bg.*\.(jpg|jpeg|png)$"])
        ref_img     = pick(files, [r"^fg.*\.(jpg|jpeg|png)$", r"^dccf.*\.jpg$"], exclude=("_mask",))
        ref_mask    = pick(files, [r"^fg.*_mask.*\.(png|jpg)$"])
        place_mask  = pick(files, [r"^mask_bg_fg.*\.(jpg|png)$"])
        if not all([bg, ref_img, place_mask]): continue

        slug    = f"{group_name[:2]}_{idx:04d}_{prompt[:60]}"
        out_dir = eedit_root / group_name / slug
        out_dir.mkdir(parents=True, exist_ok=True)
        for src in [bg, ref_img, place_mask] + ([ref_mask] if ref_mask else []):
            if src: shutil.copy2(src, out_dir / src.name)

        arr = np.array(Image.open(place_mask).convert("L"))
        ys, xs = np.where(arr > 10)
        if len(xs) == 0: continue
        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())

        entry = {
            "prompt":      prompt,
            "main_image":  str(out_dir / bg.name).replace("/content/EEdit/", "./"),
            "ref_image":   str(out_dir / ref_img.name).replace("/content/EEdit/", "./"),
            "ref_segment": str(out_dir / (ref_mask.name if ref_mask else ref_img.name)).replace("/content/EEdit/", "./"),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2
        }
        groups[group_name].append(entry)
        collected += 1

for group, imgs in groups.items():
    out_json = cfg_root / f"{group}.json"
    with open(out_json, "w") as f:
        json.dump({"imgs": imgs}, f, indent=2)
    print(f"{group}: {len(imgs)} samples -> {out_json}")


Real-Cartoon: 2 samples -> /content/EEdit/configs/composition/Real-Cartoon.json
Real-Painting: 2 samples -> /content/EEdit/configs/composition/Real-Painting.json
Real-Sketch: 2 samples -> /content/EEdit/configs/composition/Real-Sketch.json
Real-Real: 2 samples -> /content/EEdit/configs/composition/Real-Real.json


## 3. Run Composition Generation

Runs `composition_gen.py` on all 4 groups (2 images each = **8 total**).  
Output is saved to a timestamped directory so previous runs are never overwritten.

In [8]:
%cd /content/EEdit
import gc, torch, json, os
from datetime import datetime
gc.collect(); torch.cuda.empty_cache()

# Timestamped output root — prevents overwriting previous runs
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = f"EEdit_outputs/comp_{RUN_TAG}"
print(f"Output root: {OUTPUT_ROOT}")

GROUP_CFGS = [
    ("Real-Cartoon",  "RC_config.json"),
    ("Real-Painting", "RP_config.json"),
    ("Real-Sketch",   "RS_config.json"),
    ("Real-Real",     "RR_config.json"),
]

for group, cfg_file in GROUP_CFGS:
    img_cfg = f"/content/EEdit/configs/composition/{group}.json"
    if not os.path.exists(img_cfg):
        print(f"Skipping {group} -- no config"); continue
    with open(img_cfg) as f:
        n = len(json.load(f)["imgs"])
    if n == 0:
        print(f"Skipping {group} -- 0 images"); continue

    out_dir = f"{OUTPUT_ROOT}/{group}"
    print(f"\n=== {group} ({n} images) -> {out_dir} ===")
    os.makedirs(out_dir, exist_ok=True)
    !python composition_gen.py \
      --weights_dir ./weights \
      --config_path ./configs/composition/{cfg_file} \
      --img_config  ./configs/composition/{group}.json \
      --output_dir  ./{out_dir} \
      --use_predefine 1

print("\nComposition generation done.")
from pathlib import Path
total = sum(len(list(Path(f"{OUTPUT_ROOT}/{g}").glob("*.png")))
            for g, _ in GROUP_CFGS if Path(f"{OUTPUT_ROOT}/{g}").exists())
print(f"Total images generated: {total}")


/content/EEdit
Output root: EEdit_outputs/comp_20260528_101629

=== Real-Cartoon (2 images) -> EEdit_outputs/comp_20260528_101629/Real-Cartoon ===
2026-05-28 10:16:36.102693: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-28 10:16:36.167035: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
init with MyFluxAttnProcessor2_0
Loading checkpoint shards: 100% 2/2 [00:00<00:00,  3.47it/s]
Loading pipeline components...:   0% 0/5 [00:00<?, ?it/s]You set `add_prefix_space`. The tokenizer needs to be converted from the slo

In [9]:
%cd /content/EEdit
import json, shutil, os
from pathlib import Path

orig_root = OUTPUT_ROOT.replace("comp_", "comp_originals_")

# Copy original bg + ref images alongside generated outputs for comparison
for group in ["Real-Cartoon", "Real-Painting", "Real-Sketch", "Real-Real"]:
    gen_dir  = Path(f"{OUTPUT_ROOT}/{group}")
    orig_dir = Path(f"{orig_root}/{group}")
    cfg_path = f"configs/composition/{group}.json"
    if not gen_dir.is_dir() or not os.path.exists(cfg_path): continue
    orig_dir.mkdir(parents=True, exist_ok=True)
    with open(cfg_path) as f:
        cfg = json.load(f)
    for i, gf in enumerate(sorted(gen_dir.glob("*.png"))):
        if i >= len(cfg["imgs"]): break
        bg  = cfg["imgs"][i]["main_image"].lstrip("./")
        ref = cfg["imgs"][i]["ref_image"].lstrip("./")
        if os.path.exists(bg):  shutil.copy2(bg,  orig_dir / gf.name.replace(".png", "_bg.png"))
        if os.path.exists(ref): shutil.copy2(ref, orig_dir / gf.name.replace(".png", "_ref.png"))

print(f"Uploading composition results [{RUN_TAG}] to Drive...")
upload_task_results(f"composition_{RUN_TAG}", OUTPUT_ROOT, orig_root)


/content/EEdit
Uploading composition results [20260528_101629] to Drive...
Created run folder: EEdit_comp_subset_20260528_1013
View: https://drive.google.com/drive/folders/1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q
  generated/: 8 files
  originals/: 16 files
[composition_20260528_101629] uploaded 24 files total.


'1W3buOAWZjJiwPv8GxEWGFdrZwjbQ2YbR'

## 4. Evaluation Metrics

Computes background preservation (MSE / PSNR / SSIM / LPIPS) and CLIP text-image score  
for each generated image. Uses `mask_bg_fg.*` to isolate the background region.

In [10]:
%pip install -q lpips torchmetrics
print("Metric deps ready.")


Metric deps ready.


In [11]:
%cd /content/EEdit
import os, json, sys
import numpy as np
import torch
from PIL import Image
from pathlib import Path
import torchvision.transforms as T
from tqdm import tqdm

sys.path.insert(0, "/content/EEdit/img_metrics")
from calculate_ssim  import calculate_ssim
from calculate_psnr  import calculate_psnr
from calculate_lpips import calculate_lpips
from calculate_mse   import calculate_mse
from torchmetrics.multimodal.clip_score import CLIPScore

SIZE   = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

clip_metric = CLIPScore(model_name_or_path="openai/clip-vit-large-patch14").to(DEVICE)
clip_metric.eval()

def load_t(path):
    return T.ToTensor()(Image.open(path).convert("RGB").resize((SIZE, SIZE)))

def load_mask_t(path):
    return T.ToTensor()(Image.open(path).convert("L").resize((SIZE, SIZE)))

def find_mask(cfg_item, gen_file):
    """Return (mask_tensor, kind) where 1=foreground, 0=background"""
    bg_path = cfg_item["main_image"].lstrip("./")
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = os.path.join(os.path.dirname(bg_path), f"mask_bg_fg{ext}")
        if os.path.exists(candidate):
            return load_mask_t(candidate), "file"
    # Fallback: bounding box from x1/y1/x2/y2
    x1, y1, x2, y2 = cfg_item["x1"], cfg_item["y1"], cfg_item["x2"], cfg_item["y2"]
    orig_img = Image.open(bg_path)
    ow, oh = orig_img.size
    sx, sy = SIZE / ow, SIZE / oh
    mask = torch.zeros(1, SIZE, SIZE)
    mask[0, int(y1*sy):int(y2*sy), int(x1*sx):int(x2*sx)] = 1.0
    return mask, "bbox"

comp_results = {}
GROUPS = ["Real-Cartoon", "Real-Painting", "Real-Sketch", "Real-Real"]

for group in GROUPS:
    cfg_path = f"configs/composition/{group}.json"
    gen_dir  = Path(f"{OUTPUT_ROOT}/{group}")
    if not gen_dir.is_dir() or not os.path.exists(cfg_path): continue
    with open(cfg_path) as f:
        cfg = json.load(f)

    gen_files = sorted(gen_dir.glob("*.png"))
    if not gen_files:
        print(f"No outputs for {group} — skipping"); continue

    group_results = []
    print(f"\n{group} ({len(gen_files)} images):")
    for i, gf in enumerate(tqdm(gen_files)):
        if i >= len(cfg["imgs"]): break
        item = cfg["imgs"][i]
        bg_path = item["main_image"].lstrip("./")
        if not os.path.exists(bg_path): continue

        gen  = load_t(str(gf))
        orig = load_t(bg_path)
        mask_t, mtype = find_mask(item, gf)

        bg_mask  = (mask_t < 0.5).float()   # 1 = background (not the pasted region)
        orig_bg  = orig * bg_mask
        gen_bg   = gen  * bg_mask

        with torch.no_grad():
            img_uint8 = (gen.permute(1,2,0).numpy() * 255).clip(0,255).astype("uint8")
            clip_t = clip_metric(
                torch.from_numpy(img_uint8).permute(2,0,1).unsqueeze(0).to(DEVICE),
                [item["prompt"]]
            ).item()

        group_results.append({
            "file":  gf.name,
            "prompt": item["prompt"],
            "mask_source": mtype,
            "mse":   calculate_mse(orig_bg, gen_bg)["value"],
            "psnr":  calculate_psnr(orig_bg, gen_bg)["value"],
            "ssim":  calculate_ssim(orig_bg, gen_bg)["value"],
            "lpips": calculate_lpips(orig_bg, gen_bg, DEVICE)["value"][0],
            "clip":  clip_t,
        })

    n   = len(group_results)
    avg = {k: sum(r[k] for r in group_results)/n for k in ("mse","psnr","ssim","lpips","clip")}
    comp_results[group] = {"individual": group_results, "average": avg, "count": n}
    print(f'{group} ({n}): MSE={avg["mse"]:.4f}  PSNR={avg["psnr"]:.2f}  '
          f'SSIM={avg["ssim"]:.4f}  LPIPS={avg["lpips"]:.4f}  CLIP={avg["clip"]:.2f}')

all_items = [r for g in comp_results.values() for r in g["individual"]]
comp_avg  = {}
if all_items:
    n = len(all_items)
    comp_avg = {k: sum(r[k] for r in all_items)/n
                for k in ("mse","psnr","ssim","lpips","clip")}
    print(f'\nOverall ({n} images): MSE={comp_avg["mse"]:.4f}  '
          f'PSNR={comp_avg["psnr"]:.2f}  SSIM={comp_avg["ssim"]:.4f}  '
          f'LPIPS={comp_avg["lpips"]:.4f}  CLIP={comp_avg["clip"]:.2f}')

metrics_path = f"EEdit_outputs/metrics/composition_metrics_{RUN_TAG}.json"
os.makedirs("EEdit_outputs/metrics", exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump({"run_tag": RUN_TAG, "per_group": comp_results, "overall_average": comp_avg}, f, indent=2)
print(f"Saved {metrics_path}")


/content/EEdit
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [on]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100%|██████████| 233M/233M [00:01<00:00, 199MB/s]
/usr/local/lib/python3.12/dist-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle mod

Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

...

Real-Painting (2): MSE=0.0006  PSNR=31.95  SSIM=0.9128  LPIPS=0.0253  CLIP=31.02

Real-Sketch (2 images):


100%|██████████| 2/2 [00:00<00:00,  3.76it/s]


Real-Sketch (2): MSE=0.0005  PSNR=33.46  SSIM=0.9667  LPIPS=0.0131  CLIP=31.20

Real-Real (2 images):


100%|██████████| 2/2 [00:00<00:00,  3.80it/s]

Real-Real (2): MSE=0.0003  PSNR=34.62  SSIM=0.9609  LPIPS=0.0141  CLIP=28.55

Overall (8 images): MSE=0.0004  PSNR=33.73  SSIM=0.9539  LPIPS=0.0157  CLIP=30.90
Saved EEdit_outputs/metrics/composition_metrics_20260528_101629.json


In [12]:
import json, os
from pathlib import Path

# Load the metrics file for this run
metrics_path = f"EEdit_outputs/metrics/composition_metrics_{RUN_TAG}.json"
comp = json.load(open(metrics_path)) if os.path.exists(metrics_path) else None

print("=" * 65)
print("  EEdit Composition Subset Results (2 images / group)")
print("=" * 65)

if comp:
    print(f"\n{'Group':<18} {'N':>4}  {'MSE':>8}  {'PSNR':>7}  {'SSIM':>7}  {'LPIPS':>7}  {'CLIP':>7}")
    print("-" * 65)
    for group, data in comp["per_group"].items():
        a = data["average"]
        n = data["count"]
        print(f"{group:<18} {n:>4}  {a['mse']:>8.4f}  {a['psnr']:>7.2f}  {a['ssim']:>7.4f}  {a['lpips']:>7.4f}  {a['clip']:>7.2f}")
    print("-" * 65)
    a = comp["overall_average"]
    n = sum(d["count"] for d in comp["per_group"].values())
    print(f"{'OVERALL':<18} {n:>4}  {a['mse']:>8.4f}  {a['psnr']:>7.2f}  {a['ssim']:>7.4f}  {a['lpips']:>7.4f}  {a['clip']:>7.2f}")
else:
    print("composition_metrics.json not found — run the metrics cell first.")

print("\nNote: No official EEdit paper reference values for composition.")


  EEdit Composition Subset Results (2 images / group)

Group                 N       MSE     PSNR     SSIM    LPIPS     CLIP
-----------------------------------------------------------------
Real-Cartoon          2    0.0003    34.89   0.9752   0.0104    32.81
Real-Painting         2    0.0006    31.95   0.9128   0.0253    31.02
Real-Sketch           2    0.0005    33.46   0.9667   0.0131    31.20
Real-Real             2    0.0003    34.62   0.9609   0.0141    28.55
-----------------------------------------------------------------
OVERALL               8    0.0004    33.73   0.9539   0.0157    30.90

Note: No official EEdit paper reference values for composition.


## 5. Upload Metrics to Google Drive

Uploads the metrics JSON to your output Drive folder.  
View at: https://drive.google.com/drive/folders/1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q

In [13]:
import os
metrics_dir = "/content/EEdit/EEdit_outputs/metrics"
if os.path.isdir(metrics_dir):
    metrics_id = _gdrive_mkdir(drive_service, "metrics", _get_run_folder())
    n = _upload_tree(drive_service, metrics_dir, metrics_id)
    print(f"Metrics uploaded: {n} files")
else:
    print("No metrics folder found — run the metrics cell first.")

print(f"\nDone! View results:")
print(f"https://drive.google.com/drive/folders/1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q")


Metrics uploaded: 1 files

Done! View results:
https://drive.google.com/drive/folders/1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q
